# Prompt-oriented model tuning with RL

In [1]:
!pip install -U bitsandbytes

In [2]:
!pip install trl

# Task
Fine-tune a Qwen2.5 0.5B model using the ORPO algorithm on the `labeled_train_final.csv` dataset, then save the fine-tuned model and provide instructions for its inference.

## Fixing bitsandbytes in colab

_Run this section only if issues are present with bnb library_

In [ ]:
!pip install --force-reinstall bitsandbytes

  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.m

### Uninstall Conflicting Packages

Uninstall the conflicting packages `datasets` and `fsspec` to prepare for reinstalling compatible versions required for ORPO fine-tuning.


In [ ]:
!pip uninstall -y datasets
!pip uninstall -y fsspec

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: fsspec 2025.10.0
Uninstalling fsspec-2025.10.0:
  Successfully uninstalled fsspec-2025.10.0


### Reinstall Core Dependencies

Reinstall the determined compatible versions of `torch` and `numpy`. Following this, reinstall `bitsandbytes`, `transformers`, `accelerate`, and `datasets` to ensure all key libraries are installed with versions that are known to work together for the fine-tuning process. This step will also include a recommendation to restart the runtime to ensure all changes are applied correctly.


In [ ]:
print("Reinstalling specific versions of torch and numpy...")
!pip install torch==2.9.0 numpy==2.0.2

print("Reinstalling bitsandbytes, transformers, accelerate, and datasets...")
!pip install bitsandbytes transformers accelerate datasets

print("Installation complete. Please restart the Colab runtime (Runtime -> Restart runtime) to apply all changes.")

Reinstalling specific versions of torch and numpy...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.8 MB/s eta 0:00:00
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.1 MB/s eta 0:00:00
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
  Attempting uninstall: triton
    Found existing installation: triton 3.5.1
    Uninstalling triton-3.5.1:
      Successfully uninstalled triton-3.5.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5
  Attempting uninstall: torch
    Found existing installation: torch 2.9.1
    Uninstalling torch-2.9.1:
      Successfully uninstalled torch-2.9.1
ERROR: pip's dependency resolver does not currently take i

Reinstalling bitsandbytes, transformers, accelerate, and datasets...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
Installation complete. Please restart the Colab runtime (Runtime -> Restart runtime) to apply all changes.


### Verify Environment Setup

After reinstalling and restarting the runtime, verify that all critical libraries (`torch`, `numpy`, `bitsandbytes`, `transformers`, `accelerate`, `datasets`) are correctly installed and that their versions are compatible, confirming the dependency conflicts have been resolved.


In [ ]:
import torch
import numpy
import bitsandbytes
import transformers
import accelerate
import datasets

print(f"torch version: {torch.__version__}")
print(f"numpy version: {numpy.__version__}")
print(f"bitsandbytes version: {bitsandbytes.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"accelerate version: {accelerate.__version__}")
print(f"datasets version: {datasets.__version__}")

# Optional: Minimal check for bitsandbytes
try:
    from bitsandbytes.cuda_setup.main import get_compute_capability
    print(f"bitsandbytes compute capability: {get_compute_capability()}")
except Exception as e:
    print(f"Could not check bitsandbytes compute capability: {e}")

torch version: 2.9.0+cu126
numpy version: 2.0.2
bitsandbytes version: 0.48.2
transformers version: 4.57.1
accelerate version: 1.11.0
datasets version: 4.0.0
Could not check bitsandbytes compute capability: No module named 'bitsandbytes.cuda_setup'


## Load Qwen2.5 Model and Tokenizer

Load the pre-trained Qwen2.5 0.5B model and its corresponding tokenizer from Hugging Face Transformers. Ensure the tokenizer is configured correctly for the model, including padding and special tokens. This step assumes the environment is now stable.


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 2. Define the pre-trained model name
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'

# 3. Create a BitsAndBytesConfig object for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 4. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")

# tokenizer = tokenizer(
#     [orpo_dataset[0]["chosen"], orpo_dataset[0]["rejected"]],
#     padding=True,
#     truncation=True,
#     return_tensors="pt"
# )


# 5. Load the pre-trained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

# 6. Print the loaded model and tokenizer
print("Loaded Model:")
print(model)
print("\nLoaded Tokenizer:")
print(tokenizer)

Loaded Model:
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm(

## Load Dataset

Load the `labeled_train_final.csv` file into a pandas DataFrame and inspect its structure to understand the columns containing instructions and improved instructions.


In [4]:
import pandas as pd

# Load the labeled_train_final.csv file into a pandas DataFrame
df = pd.read_json('/content/drive/MyDrive/Datasets/NLP-RL/rlhf_train.json')

# Display the first few rows of the DataFrame to inspect the data
print("First 5 rows of the DataFrame:")
print(df.head())

# Print the DataFrame's information, including column names and data types
print("\nDataFrame Info:")
df.info()

# List the columns present in the DataFrame
print("\nColumns in the DataFrame:")
print(df.columns.tolist())

First 5 rows of the DataFrame:
    qid                                    original_prompt  \
0  3841  How can microbial biotechnology be utilized to...   
1  3842  How can microbial biotechnology be used to pro...   
2  3845  How can microbial biotechnology be utilized to...   
3  3846  How is microbial biotechnology used in the pro...   
4  3847  How can the production of antibiotics through ...   

                                         best_prompt  \
0  Produce a step-by-step plan to address the fol...   
1  How can microbial biotechnology improve antibi...   
2  Evaluate the use of microbial biotechnology in...   
3  Produce a step-by-step guide on the applicatio...   
4  To optimize antibiotic production via microbia...   

                                        worst_prompt  
0  Explain how microbial biotechnology can be use...  
1  A team of biotechnologists aims to develop a n...  
2  Consider the production of a common antibiotic...  
3  Describe how microbial biotechnology

### Prepare Dataset for ORPO

Preprocess the loaded dataset to format it appropriately for ORPO training, creating 'prompt', 'chosen', and 'rejected' columns based on the 'instructions', 'improved instructions', 'original_prompt', and 'context' data.


In [5]:
def format_prompt(prompt, example=None):
    system_prompt = "You are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering."

    chat_input = [{"role": "system", "content": system_prompt},
         {"role": "user", "content": prompt}]

    if example != None:
        chat_input.append({"role": "assistant", "content": example})

    return tokenizer.apply_chat_template(
        chat_input,
        tokenize=False,
        add_generation_prompt=False
    )

In [6]:
df_processed = df.copy()

# 1. Remove rows from the DataFrame `df` where the `instruction` column has missing (NaN) values
df_processed.dropna(subset=['original_prompt','best_prompt','worst_prompt'], inplace=True)

# Define the system prompt to assert the model's role

# Prepend the system prompt to the 'prompt' column
df_processed['prompt'] = df_processed['original_prompt'].apply(lambda x: format_prompt(x))

# 3. Create a new column named `chosen`
df_processed['chosen'] = df_processed.apply(lambda row: format_prompt(row['original_prompt'], row['best_prompt']), axis=1)

# 4. Create a new column named `rejected`
df_processed['rejected'] = df_processed.apply(lambda row: format_prompt(row['original_prompt'], row['worst_prompt']), axis=1)

# 5. Select only the `prompt`, `chosen`, and `rejected` columns
df_orpo = df_processed[['prompt', 'chosen', 'rejected']]

print("First 5 rows of the ORPO-ready DataFrame with system prompt:")
print(df_orpo.head())

First 5 rows of the ORPO-ready DataFrame with system prompt:
                                              prompt  \
0  <|im_start|>system\nYou are an expert LLM prom...   
1  <|im_start|>system\nYou are an expert LLM prom...   
2  <|im_start|>system\nYou are an expert LLM prom...   
3  <|im_start|>system\nYou are an expert LLM prom...   
4  <|im_start|>system\nYou are an expert LLM prom...   

                                              chosen  \
0  <|im_start|>system\nYou are an expert LLM prom...   
1  <|im_start|>system\nYou are an expert LLM prom...   
2  <|im_start|>system\nYou are an expert LLM prom...   
3  <|im_start|>system\nYou are an expert LLM prom...   
4  <|im_start|>system\nYou are an expert LLM prom...   

                                            rejected  
0  <|im_start|>system\nYou are an expert LLM prom...  
1  <|im_start|>system\nYou are an expert LLM prom...  
2  <|im_start|>system\nYou are an expert LLM prom...  
3  <|im_start|>system\nYou are an expert LLM 

In [7]:
from datasets import Dataset

# Convert the pandas DataFrame to a Hugging Face Dataset
orpo_dataset = Dataset.from_pandas(df_orpo)

# Print the first few examples of the Hugging Face Dataset
print("First 5 examples of the Hugging Face Dataset:")
print(orpo_dataset.select(range(5)))
print(len(orpo_dataset))

First 5 examples of the Hugging Face Dataset:
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 5
})
4211


In [ ]:
print(tokenizer.pad_token_id)
print(tokenizer.eos_token_id)

151643
151645


Add [pad] and [eos] tokens

In [ ]:
if not tokenizer.pad_token_id or not tokenizer.eos_token_id:
  print("Adding pad and eos tokens")
  tokenizer.add_special_tokens({"pad_token": "[PAD]"})
  tokenizer.add_special_tokens({"eos_token": "[EOS]"})

  model.resize_token_embeddings(len(tokenizer))
  model.config.pad_token_id = tokenizer.pad_token_id
  model.config.eos_token_id = tokenizer.eos_token_id
else:
  print("Tokens already exist")

Tokens already exist


## Configure ORPO Training

### Subtask:
Set up the ORPO training arguments, including parameters like learning rate, batch size, number of epochs, ORPO-specific beta parameter, and any other relevant configurations for efficient and effective fine-tuning.


**Reasoning**:
To set up the ORPO training arguments, I need to import the necessary classes, define an output directory, and then instantiate `TrainingArguments` and `ORPOConfig` with the specified parameters.



In [ ]:
from trl import ORPOConfig, ORPOTrainer
from transformers import TrainingArguments

# 2. Define the output directory
output_dir = "./qwen2.5-0.5b-orpo-finetuned"

# 3. Instantiate TrainingArguments
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    # max_steps=1000, # Removed to allow num_train_epochs to control training duration
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_steps=200,
    logging_steps=50,
    output_dir=output_dir,
    report_to="none", # Disable logging to external services for simplicity
    remove_unused_columns=False,

)

# 4. Instantiate ORPOConfig
orpo_config = ORPOConfig(
    beta=0.1, # ORPO specific beta parameter
    learning_rate=5e-5,
    max_length=2048, # Max sequence length for tokenizer
    max_prompt_length=512,
    per_device_train_batch_size=training_args.per_device_train_batch_size,
    gradient_accumulation_steps=training_args.gradient_accumulation_steps,
    output_dir=output_dir,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=0,
    num_train_epochs=2,
    # max_steps=-1, # Removed to allow num_train_epochs to control training duration
    save_steps=training_args.save_steps,
    remove_unused_columns=False,
    logging_steps=training_args.logging_steps,
)

# 5. Print the configured TrainingArguments and ORPOConfig
print("Configured TrainingArguments:")
print(training_args)
print("\nConfigured ORPOConfig:")
print(orpo_config)


Configured TrainingArguments:
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
e

## Initialize and Train ORPO Trainer

### Subtask:
Initialize the ORPO trainer with the prepared dataset, loaded model, tokenizer, and defined training arguments. Then, start the training process to fine-tune the Qwen2.5 model using the ORPO algorithm.


In [ ]:
orpo_dataset[0]

{'prompt': "<|im_start|>system\nYou are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering.<|im_end|>\n<|im_start|>user\nHow can microbial biotechnology be utilized to produce pharmaceuticals more efficiently while ensuring safety and quality standards?<|im_end|>\n",
 'chosen': "<|im_start|>system\nYou are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering.<|im_end|>\n<|im_start|>user\nHow can microbial biotechnology be utilized to produce pharmaceuticals more efficiently while ensuring safety and quality standards?<|im_end|>\n<|im_start|>assistant\nProduce a step-by-step plan to address the following scenario: A pharmaceutical company wishes

In [ ]:
# Validate dataset formatting
for i, row in enumerate(orpo_dataset):
    try:
        enc = tokenizer(
            [row["chosen"], row["rejected"]],
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
    except Exception as e:
        print("Error at index", i)
        print(row)
        print("Exception:", e)
        break


In [ ]:
from peft import LoraConfig, get_peft_model

# Configure LoRA
lora_config = LoraConfig(
    r=16,  # LoRA attention dimension
    lora_alpha=32,  # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA layers
    task_type="CAUSAL_LM",  # Task type
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

empty_eval = orpo_dataset.select([])

orpo_trainer = ORPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=orpo_config,
    train_dataset=orpo_dataset,
    eval_dataset=empty_eval,
)
print("ORPO Trainer initialized.")

/usr/local/lib/python3.12/dist-packages/trl/trainer/orpo_trainer.py:153: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(


Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

ORPO Trainer initialized.


In [ ]:
print("Starting ORPO training...")
orpo_trainer.train()
print("ORPO training complete.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting ORPO training...


Step,Training Loss
50,1.040500
100,0.788900
150,0.784800
200,0.772500
250,0.760300
300,0.760200
350,0.737100
400,0.727900
450,0.723000
500,0.723600


ORPO training complete.


In [ ]:
print("Saving fine-tuned model...")
orpo_trainer.save_model()
print(f"Model saved to {output_dir}")

Saving fine-tuned model...
Model saved to ./qwen2.5-0.5b-orpo-finetuned


Zip output folder for download

In [ ]:
import shutil
import os

# Define the output directory for the fine-tuned model
output_dir = "./qwen2.5-0.5b-orpo-finetuned"

# Define the name for the zip file
zip_file_name = "qwen2.5-0.5b-orpo-finetuned"

# Create the zip archive
shutil.make_archive(zip_file_name, 'zip', output_dir)

print(f"Model output directory '{output_dir}' has been zipped to '{zip_file_name}.zip'")

# You can verify the created zip file
if os.path.exists(f"{zip_file_name}.zip"):
    print(f"File '{zip_file_name}.zip' created successfully.")
else:
    print(f"Failed to create '{zip_file_name}.zip'.")

Model output directory './qwen2.5-0.5b-orpo-finetuned' has been zipped to 'qwen2.5-0.5b-orpo-finetuned.zip'
File 'qwen2.5-0.5b-orpo-finetuned.zip' created successfully.


In [ ]:
import os

file_path = "./qwen2.5-0.5b-orpo-finetuned.zip"

if os.path.exists(file_path):
    file_size_bytes = os.path.getsize(file_path)
    file_size_kb = file_size_bytes / 1024
    file_size_mb = file_size_kb / 1024
    print(f"The size of '{file_path}' is:")
    print(f"- {file_size_bytes} bytes")
    print(f"- {file_size_kb:.2f} KB")
    print(f"- {file_size_mb:.2f} MB")
else:
    print(f"File '{file_path}' not found.")

The size of './qwen2.5-0.5b-orpo-finetuned.zip' is:
- 356476268 bytes
- 348121.36 KB
- 339.96 MB


Save to drive

In [ ]:
import shutil
import os

# Define the source file path (the zipped model)
source_file_path = "./qwen2.5-0.5b-orpo-finetuned.zip"

# Define the destination folder in Google Drive
destination_folder = "/content/drive/MyDrive/Datasets/NLP-RL"

# Define the full destination path for the file
destination_file_path = os.path.join(destination_folder, os.path.basename(source_file_path))

# Copy the file
try:
    shutil.copy(source_file_path, destination_file_path)
    print(f"File '{source_file_path}' successfully copied to '{destination_file_path}'")
except FileNotFoundError:
    print(f"Error: Source file '{source_file_path}' not found.")
except Exception as e:
    print(f"An error occurred while copying the file: {e}")

File './qwen2.5-0.5b-orpo-finetuned.zip' successfully copied to '/content/drive/MyDrive/Datasets/NLP-RL/qwen2.5-0.5b-orpo-finetuned.zip'


## Inference

### Subtask:
To load the fine-tuned model and use it for inference:


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Define the path to your saved model
saved_model_path = "./qwen2.5-0.5b-orpo-finetuned"

# Load the base model
base_model_name = 'Qwen/Qwen2.5-0.5B-Instruct'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, padding_side="left")
tokenizer.add_special_tokens({"pad_token": "[PAD]", "eos_token": "[EOS]"})

# Load the fine-tuned model with the PEFT adapters
# Ensure you load the base model first, then apply the adapters
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16, # Or whatever dtype you used during training
    device_map='auto',
    trust_remote_code=True
)

# Crucial step: Resize token embeddings to match the tokenizer's new vocabulary size
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

# Load the LoRA adapters from the saved directory
model = PeftModel.from_pretrained(model, saved_model_path)
model.eval() # Set model to evaluation mode

print("Fine-tuned model loaded successfully for inference.")

Fine-tuned model loaded successfully for inference.


Generate sample output

In [4]:
# Example of how to generate text
def generate_text(prompt, max_new_tokens=108):
    system_prompt = "You are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering."

    chat_input = [{"role": "system", "content": system_prompt},
         {"role": "user", "content": prompt}]

    # Tokenize the chat input
    tokenized_chat = tokenizer.apply_chat_template(
        chat_input,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt" # Return as PyTorch tensor directly
    )

    input_ids = tokenized_chat.to(model.device)

    output = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.3,
        pad_token_id=tokenizer.pad_token_id, # Ensure pad_token_id is used during generation
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode only the generated part (after the input prompt)
    # The generated output contains the input_ids as a prefix
    generated_text = tokenizer.decode(output[0][len(input_ids[0]):], skip_special_tokens=True)
    return generated_text


In [5]:
# Test the model with a prompt
example_prompt = "Develop a creative narrative about neural prosthetics"
generated_response = generate_text(example_prompt)
print("\n--- Generated Response ---")
print(generated_response)


--- Generated Response ---
Certainly! Here's a creative narrative for a story involving neural prosthetics:

---

In a world where artificial intelligence has advanced beyond human capability, a group of scientists and engineers have developed a revolutionary new technology called Neural Prosthetics. These devices allow individuals with complete paralysis or severe brain damage to regain some level of function through the use of electrodes implanted in their brains.

The protagonist, Dr. Sarah Lee, is one such individual. She was born without any limbs, but her mind had been restored by a groundbreaking procedure that involved creating a network


### Inference from dataset

In [6]:
import pandas as pd

# Load the labeled_train_final.csv file into a pandas DataFrame
df = pd.read_json('/content/drive/MyDrive/Datasets/NLP-RL/rlhf_test.json')

# Display the first few rows of the DataFrame to inspect the data
print("First 5 rows of the DataFrame:")
print(df.head())

qa_df = []

for i, row in df.iterrows():
  qa_df.append({'qid':row['qid'],'original_prompt':row['original_prompt']})

print(len(qa_df))
qa_df[:5]

First 5 rows of the DataFrame:
    qid                                    original_prompt  \
0  2880  How has the evolution of bioluminescence in ma...   
1  2881  How has the ability to produce and use biolumi...   
2  2882  What are the possible evolutionary advantages ...   
3  2883  How has the evolution of bioluminescence in ma...   
4  2884  How has bioluminescence evolved in marine orga...   

                                         best_prompt  \
0  How has bioluminescence evolved in marine orga...   
1  What selective pressures have driven the evolu...   
2  Consider a marine environment where biolumines...   
3  How has the evolution of bioluminescence in ma...   
4  Explain the evolutionary process of biolumines...   

                                        worst_prompt  
0  Examine the evolution of bioluminescence in ma...  
1  Examine the evolutionary origins and diversifi...  
2  Explain the evolutionary advantages of biolumi...  
3  Explain how the evolution of biolumi

[{'qid': 2880,
  'original_prompt': 'How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems?'},
 {'qid': 2881,
  'original_prompt': 'How has the ability to produce and use bioluminescence evolved in marine organisms? What selective pressures and adaptations have led to the development and diversity of bioluminescent structures and behaviors?'},
 {'qid': 2882,
  'original_prompt': 'What are the possible evolutionary advantages of bioluminescence in marine organisms and how has this adaptation enabled them to survive in their ecosystem?'},
 {'qid': 2883,
  'original_prompt': 'How has the evolution of bioluminescence in marine organisms impacted their survival and reproduction over time?'},
 {'qid': 2884,
  'original_prompt': 'How has bioluminescence evolved in marine organisms and what are the advantages and disadvantages of this adaptation in different environments?'}]

In [7]:
def preprocess_prompts(prompt, max_new_tokens=108):
    system_prompt = "You are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering."

    chat_input = [system_prompt, prompt]

    return chat_input

In [8]:
prompts = [preprocess_prompts(q['original_prompt']) for q in qa_df]

In [11]:
print(prompts[:5])

[["You are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering.", 'How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems?'], ["You are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' into a good prompt, adhering to best practices of prompt engineering.", 'How has the ability to produce and use bioluminescence evolved in marine organisms? What selective pressures and adaptations have led to the development and diversity of bioluminescent structures and behaviors?'], ["You are an expert LLM prompt engineer. Your goal is to generate clear, concise, and effective prompts for other large language models. Refine the 'Task Description' 

In [9]:
from tqdm import tqdm

batch_size = 16  # whatever fits in VRAM
batches = [prompts[i:i+batch_size] for i in range(0, len(prompts), batch_size)]

results = []

for batch in tqdm(batches):
    enc = tokenizer(batch, padding=True, return_tensors="pt").to(model.device)

    gen = model.generate(
        **enc,
        max_new_tokens=108,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    # decode each row separately to avoid shrinkage
    decoded = [
        tokenizer.decode(row, skip_special_tokens=True)
        for row in gen
    ]

    # remove prompt from generation (Qwen echoes prompt)
    cleaned = []
    for inp, full in zip(batch, decoded):
        # safe trimming — avoids mismatches
        cleaned.append(full[len(inp[0]):].strip())

    for inp, out in zip(batch, cleaned):
        results.append({
            "input": inp[1],
            "output": out
        })

for result in results[:5]:
  print('---\n',result['input'],'\n',result['output'])

print(len(results))

100%|██████████| 47/47 [07:47<00:00,  9.95s/it]

---
 How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems? 
 How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems? Please provide a detailed explanation of how this process involves various biological processes such as photosynthesis, cellular respiration, and enzyme activity. Additionally, please include examples of specific species that have demonstrated this ability through their unique adaptations. Finally, discuss the potential implications of this research on future conservation efforts and ecosystem management strategies.

The evolution of bioluminescence in marine organisms has been a fascinating area of study due to its role in survival and success in their respective ecosystems. This process involves several biological processes including photosynthesis, cellular respiration, and enzyme
---
 How has the ability to pr

In [10]:
for result in results[:5]:
  print('---\n',result['input'],'\n',result['output'])

---
 How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems? 
 How has the evolution of bioluminescence in marine organisms contributed to their survival and success in their respective ecosystems? Please provide a detailed explanation of how this process involves various biological processes such as photosynthesis, cellular respiration, and enzyme activity. Additionally, please include examples of specific species that have demonstrated this ability through their unique adaptations. Finally, discuss the potential implications of this research on future conservation efforts and ecosystem management strategies.

The evolution of bioluminescence in marine organisms has been a fascinating area of study due to its role in survival and success in their respective ecosystems. This process involves several biological processes including photosynthesis, cellular respiration, and enzyme
---
 How has the ability to pr

In [ ]:
from tqdm import tqdm

for q in tqdm(qa_df):
  model_prompt = generate_text(q['original_prompt'])
  q['generated_prompt'] = model_prompt
  # print("\nOriginal prompt: ", q['original_prompt'])
  # print("\n Generated prompt: ", model_prompt)

  1%|▏         | 10/743 [01:09<1:25:21,  6.99s/it]


KeyboardInterrupt: 

In [12]:
import json

json_path = '/content/drive/MyDrive/Datasets/NLP-RL/full_base_gen_prompts.json'
with open(json_path, 'w') as f:
  json.dump(results, f)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
